# Module 5: How to Trust Your Results

In the previous modules, we used a single 80/20 train/test split. But how reliable is that?
What if we got lucky (or unlucky) with the split? **Cross-validation** gives us a more trustworthy answer.

We'll also confirm which model truly wins this competition.

In [ ]:
#@title Setup { display-mode: "form" }

import sys; sys.path.insert(0, '.')
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import Button, VBox, HBox, Output, HTML, IntSlider, Dropdown
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

from helpers import load_dataset, apply_dark_theme, takeaway_box, metric_card
from helpers.styling import (
    role_legend_html, styled_button,
    DARK_BG, AXES_BG, TEXT_COLOR, GREEN, RED, GRAY, MUTED_TEXT, ACCENT_BLUE
)

X, y, codebook, role_map = load_dataset()
print(f'Loaded: {X.shape[0]} countries x {X.shape[1]} features')

---
## 1. Split Instability: How Much Does the Random Seed Matter?

A single train/test split depends on random chance. Let's see how much model performance
varies across many different splits.

In [ ]:
#@title Split Instability Widget { display-mode: "form" }

class SplitInstabilityWidget:
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.output = Output()

    def create_ui(self):
        self.n_seeds_slider = IntSlider(
            value=20, min=5, max=50, step=5,
            description='Seeds:', style={'description_width': '80px'},
            layout=widgets.Layout(width='350px')
        )
        btn = styled_button('Run Multiple Splits', width='180px')
        btn.on_click(self.run)

        ui = VBox([
            HBox([self.n_seeds_slider, btn]),
            self.output
        ])
        display(ui)

    def run(self, b):
        n_seeds = self.n_seeds_slider.value
        with self.output:
            clear_output(wait=True)
            print(f'Training 4 models x {n_seeds} splits... please wait.')

        models_cfg = {
            'OLS': lambda: LinearRegression(),
            'Ridge': lambda: Ridge(alpha=100),
            'Lasso': lambda: Lasso(alpha=100, max_iter=5000),
            'Random Forest': lambda: RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
        }
        colors = {'OLS': '#e2e8f0', 'Ridge': ACCENT_BLUE, 'Lasso': '#f59e0b', 'Random Forest': GREEN}

        results = {name: [] for name in models_cfg}
        for seed in range(n_seeds):
            X_tr, X_te, y_tr, y_te = train_test_split(self.X, self.y, test_size=0.2, random_state=seed)
            scaler = StandardScaler().fit(X_tr)
            X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)
            for name, make_model in models_cfg.items():
                model = make_model()
                if name == 'Random Forest':
                    model.fit(X_tr, y_tr)
                    score = r2_score(y_te, model.predict(X_te))
                else:
                    model.fit(X_tr_s, y_tr)
                    score = r2_score(y_te, model.predict(X_te_s))
                results[name].append(score)

        with self.output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(12, 6), facecolor=DARK_BG)
            apply_dark_theme(fig, ax)

            names = list(models_cfg.keys())
            data = [results[n] for n in names]
            bp = ax.boxplot(data, labels=names, patch_artist=True, widths=0.5,
                           medianprops=dict(color='white', linewidth=2),
                           whiskerprops=dict(color=MUTED_TEXT),
                           capprops=dict(color=MUTED_TEXT),
                           flierprops=dict(markeredgecolor=MUTED_TEXT, markersize=4))
            for patch, name in zip(bp['boxes'], names):
                patch.set_facecolor(colors[name])
                patch.set_alpha(0.7)
                patch.set_edgecolor('white')

            ax.axhline(y=0, color=RED, linestyle='--', alpha=0.5, label='R² = 0')
            ax.set_ylabel('Test R²', fontsize=13, fontweight='bold')
            ax.set_title(f'Test R² Across {n_seeds} Random Splits', fontsize=15, fontweight='bold')
            ax.tick_params(axis='x', labelsize=12)
            ax.legend(facecolor=AXES_BG, edgecolor='#4a4a6a', labelcolor=TEXT_COLOR)

            # Add median annotations
            for i, name in enumerate(names):
                med = np.median(results[name])
                ax.text(i + 1, med + 0.02, f'{med:.2f}', ha='center', va='bottom',
                        color='white', fontsize=11, fontweight='bold')

            plt.tight_layout()
            plt.show()

            # Summary
            summary_html = '<div style="display: flex; gap: 10px; flex-wrap: wrap; margin-top: 10px;">'
            for name in names:
                vals = results[name]
                med = np.median(vals)
                iqr = np.percentile(vals, 75) - np.percentile(vals, 25)
                c = colors[name]
                summary_html += f'''
                <div style="background: #1e293b; padding: 12px; border-radius: 8px;
                            border: 1px solid #475569; min-width: 160px; text-align: center;">
                    <div style="color: {c}; font-weight: bold; font-size: 14px;">{name}</div>
                    <div style="color: #e2e8f0; font-size: 13px; margin-top: 4px;">Median: {med:.3f}</div>
                    <div style="color: #94a3b8; font-size: 12px;">IQR: {iqr:.3f}</div>
                </div>'''
            summary_html += '</div>'
            display(HTML(summary_html))

In [ ]:
#@title Launch Split Instability Demo { display-mode: "form" }
split_widget = SplitInstabilityWidget(X, y)
split_widget.create_ui()

---
## 2. Cross-Validation Explorer

Instead of relying on a single split, **k-fold cross-validation** rotates through k different
validation sets, training on the rest each time. This gives k estimates of performance.

In [ ]:
#@title CV Explorer Widget { display-mode: "form" }

class CVExplorerWidget:
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.output = Output()

    def create_ui(self):
        self.k_slider = widgets.SelectionSlider(
            options=[3, 5, 10], value=5,
            description='Folds (k):', style={'description_width': '80px'},
            layout=widgets.Layout(width='250px')
        )
        self.model_dropdown = Dropdown(
            options=['OLS', 'Ridge (a=100)', 'Lasso (a=100)', 'Random Forest'],
            value='Random Forest', description='Model:',
            style={'description_width': '80px'},
            layout=widgets.Layout(width='280px')
        )
        btn = styled_button('Run Cross-Validation', width='180px')
        btn.on_click(self.run)

        ui = VBox([
            HBox([self.k_slider, self.model_dropdown, btn]),
            self.output
        ])
        display(ui)

    def _make_pipeline(self, name):
        if name == 'OLS':
            return Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
        elif name == 'Ridge (a=100)':
            return Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=100))])
        elif name == 'Lasso (a=100)':
            return Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=100, max_iter=5000))])
        else:
            return RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)

    def run(self, b):
        k = self.k_slider.value
        name = self.model_dropdown.value
        pipe = self._make_pipeline(name)

        with self.output:
            clear_output(wait=True)
            print(f'Running {k}-fold CV for {name}...')

        scores = cross_val_score(pipe, self.X, self.y, cv=k, scoring='r2')

        with self.output:
            clear_output(wait=True)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor=DARK_BG,
                                           gridspec_kw={'width_ratios': [3, 1]})
            apply_dark_theme(fig, [ax1, ax2])

            # Bar chart of fold scores
            fold_labels = [f'Fold {i+1}' for i in range(k)]
            bar_colors = [GREEN if s > 0 else RED for s in scores]
            bars = ax1.bar(fold_labels, scores, color=bar_colors, alpha=0.8, edgecolor='white', linewidth=0.5)
            ax1.axhline(y=scores.mean(), color=ACCENT_BLUE, linestyle='--', linewidth=2,
                        label=f'Mean: {scores.mean():.3f}')
            ax1.axhline(y=0, color=RED, linestyle=':', alpha=0.4)

            for bar, score in zip(bars, scores):
                ypos = score + 0.01 if score >= 0 else score - 0.03
                ax1.text(bar.get_x() + bar.get_width()/2, ypos, f'{score:.3f}',
                         ha='center', va='bottom' if score >= 0 else 'top',
                         color='white', fontsize=11, fontweight='bold')

            ax1.set_ylabel('R²', fontsize=13, fontweight='bold')
            ax1.set_title(f'{name} — {k}-Fold Cross-Validation', fontsize=14, fontweight='bold')
            ax1.legend(facecolor=AXES_BG, edgecolor='#4a4a6a', labelcolor=TEXT_COLOR, fontsize=11)

            # Fold diagram
            ax2.set_xlim(0, 1)
            ax2.set_ylim(-0.5, k - 0.5)
            ax2.set_title('Fold Structure', fontsize=13, fontweight='bold')
            ax2.set_yticks(range(k))
            ax2.set_yticklabels([f'Split {i+1}' for i in range(k)])
            ax2.set_xticks([])

            for i in range(k):
                fold_width = 1.0 / k
                for j in range(k):
                    color = '#ef4444' if j == i else '#22c55e'
                    label_text = 'Val' if j == i else ''
                    rect = plt.Rectangle((j * fold_width, i - 0.35), fold_width - 0.01, 0.7,
                                         facecolor=color, alpha=0.6, edgecolor='white', linewidth=0.5)
                    ax2.add_patch(rect)

            # Legend for fold diagram
            from matplotlib.patches import Patch
            ax2.legend(
                handles=[Patch(facecolor='#22c55e', alpha=0.6, label='Train'),
                         Patch(facecolor='#ef4444', alpha=0.6, label='Validate')],
                loc='lower center', facecolor=AXES_BG, edgecolor='#4a4a6a',
                labelcolor=TEXT_COLOR, fontsize=10
            )

            plt.tight_layout()
            plt.show()

            # Metric cards
            cards = HBox([
                metric_card('CV Mean R²', f'{scores.mean():.4f}', GREEN if scores.mean() > 0.3 else RED),
                metric_card('CV Std', f'{scores.std():.4f}', ACCENT_BLUE),
                metric_card('Min Fold', f'{scores.min():.4f}', RED),
                metric_card('Max Fold', f'{scores.max():.4f}', GREEN),
            ])
            display(cards)

In [ ]:
#@title Launch CV Explorer { display-mode: "form" }
cv_widget = CVExplorerWidget(X, y)
cv_widget.create_ui()

---
## 3. Final Model Dashboard

Let's bring all models together for the definitive comparison: single-split performance,
cross-validated performance, and overfitting gap.

In [ ]:
#@title Final Dashboard Widget { display-mode: "form" }

class FinalDashboardWidget:
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.output = Output()

    def create_ui(self):
        btn = styled_button('Run Full Comparison', color='#16a34a', width='200px')
        btn.on_click(self.run)
        ui = VBox([btn, self.output])
        display(ui)

    def run(self, b):
        with self.output:
            clear_output(wait=True)
            print('Training 5 models with cross-validation... please wait.')

        X_tr, X_te, y_tr, y_te = train_test_split(self.X, self.y, test_size=0.2, random_state=42)
        scaler = StandardScaler().fit(X_tr)
        X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

        configs = [
            ('OLS', LinearRegression(), True),
            ('Ridge (a=100)', Ridge(alpha=100), True),
            ('Lasso (a=100)', Lasso(alpha=100, max_iter=5000), True),
            ('ElasticNet (a=10)', ElasticNet(alpha=10, max_iter=5000), True),
            ('Random Forest (500)', RandomForestRegressor(n_estimators=500, max_depth=15, random_state=42, n_jobs=-1), False),
        ]

        rows = []
        for name, model, needs_scale in configs:
            if needs_scale:
                model.fit(X_tr_s, y_tr)
                train_r2 = r2_score(y_tr, model.predict(X_tr_s))
                test_r2 = r2_score(y_te, model.predict(X_te_s))
                pipe = Pipeline([('scaler', StandardScaler()), ('model', model.__class__(**model.get_params()))])
                cv = cross_val_score(pipe, self.X, self.y, cv=5, scoring='r2')
                n_nonzero = np.sum(np.abs(model.coef_) > 1e-6) if hasattr(model, 'coef_') else 'N/A'
            else:
                model.fit(X_tr, y_tr)
                train_r2 = r2_score(y_tr, model.predict(X_tr))
                test_r2 = r2_score(y_te, model.predict(X_te))
                cv = cross_val_score(model, self.X, self.y, cv=5, scoring='r2')
                n_nonzero = self.X.shape[1]

            rows.append({
                'name': name, 'train_r2': train_r2, 'test_r2': test_r2,
                'cv_mean': cv.mean(), 'cv_std': cv.std(),
                'overfit': train_r2 - test_r2, 'n_nonzero': n_nonzero
            })

        with self.output:
            clear_output(wait=True)

            # Results table
            table_html = '''
            <table style="border-collapse: collapse; width: 100%; font-size: 14px; margin: 10px 0;">
            <tr style="background: #1e293b; border-bottom: 2px solid #475569;">
                <th style="padding: 10px; color: #e0e7ff; text-align: left;">Model</th>
                <th style="padding: 10px; color: #e0e7ff; text-align: center;">Train R²</th>
                <th style="padding: 10px; color: #e0e7ff; text-align: center;">Test R²</th>
                <th style="padding: 10px; color: #e0e7ff; text-align: center;">CV R² (mean±std)</th>
                <th style="padding: 10px; color: #e0e7ff; text-align: center;">Overfit Gap</th>
                <th style="padding: 10px; color: #e0e7ff; text-align: center;">Features Used</th>
            </tr>'''

            for r in rows:
                is_rf = 'Random Forest' in r['name']
                bg = '#0f3a1a' if is_rf else '#0f172a'
                border = f'border-left: 3px solid {GREEN};' if is_rf else ''
                test_color = GREEN if r['test_r2'] > 0.3 else (RED if r['test_r2'] < 0 else '#f59e0b')
                overfit_color = GREEN if abs(r['overfit']) < 0.3 else RED
                nz = str(r['n_nonzero'])

                table_html += f'''
                <tr style="background: {bg}; {border} border-bottom: 1px solid #334155;">
                    <td style="padding: 10px; color: #e0e7ff; font-weight: {'bold' if is_rf else 'normal'};">{r['name']}</td>
                    <td style="padding: 10px; color: #94a3b8; text-align: center;">{r['train_r2']:.4f}</td>
                    <td style="padding: 10px; color: {test_color}; text-align: center; font-weight: bold;">{r['test_r2']:.4f}</td>
                    <td style="padding: 10px; color: #e0e7ff; text-align: center;">{r['cv_mean']:.4f} ± {r['cv_std']:.4f}</td>
                    <td style="padding: 10px; color: {overfit_color}; text-align: center;">{r['overfit']:.4f}</td>
                    <td style="padding: 10px; color: #94a3b8; text-align: center;">{nz}</td>
                </tr>'''

            table_html += '</table>'
            display(HTML(table_html))

            # Grouped bar chart
            fig, ax = plt.subplots(figsize=(14, 6), facecolor=DARK_BG)
            apply_dark_theme(fig, ax)

            names = [r['name'] for r in rows]
            x = np.arange(len(names))
            w = 0.25

            train_vals = [r['train_r2'] for r in rows]
            test_vals = [r['test_r2'] for r in rows]
            cv_vals = [r['cv_mean'] for r in rows]

            ax.bar(x - w, train_vals, w, label='Train R²', color='#94a3b8', alpha=0.7, edgecolor='white', linewidth=0.5)
            ax.bar(x, test_vals, w, label='Test R²', color=ACCENT_BLUE, alpha=0.8, edgecolor='white', linewidth=0.5)
            ax.bar(x + w, cv_vals, w, label='CV R² (mean)', color=GREEN, alpha=0.8, edgecolor='white', linewidth=0.5)

            ax.set_xticks(x)
            ax.set_xticklabels([n.replace(' (', '\n(') for n in names], fontsize=11)
            ax.set_ylabel('R²', fontsize=13, fontweight='bold')
            ax.set_title('Final Model Comparison', fontsize=15, fontweight='bold')
            ax.legend(facecolor=AXES_BG, edgecolor='#4a4a6a', labelcolor=TEXT_COLOR, fontsize=11)
            ax.axhline(y=0, color=RED, linestyle=':', alpha=0.4)

            plt.tight_layout()
            plt.show()

            # Executive summary
            rf_row = [r for r in rows if 'Random Forest' in r['name']][0]
            summary = f'''
            <div style="background: linear-gradient(135deg, #14532d, #0f3a1a); padding: 20px 25px;
                        border-radius: 12px; border: 2px solid {GREEN}; margin-top: 15px; max-width: 900px;">
                <div style="color: {GREEN}; font-size: 18px; font-weight: bold; margin-bottom: 10px;">Winner: Random Forest</div>
                <div style="color: #e0e7ff; font-size: 14px; line-height: 1.8;">
                    Best test performance (R² = {rf_row['test_r2']:.4f}), most stable across splits,
                    and best at identifying causal features (52% of importance on just 22 causal features).
                </div>
            </div>'''
            display(HTML(summary))

In [ ]:
#@title Launch Final Dashboard { display-mode: "form" }
dashboard = FinalDashboardWidget(X, y)
dashboard.create_ui()

---
## 4. Lessons Learned

Here's what we've discovered across all five modules.

In [ ]:
#@title Lessons Learned { display-mode: "form" }

lessons = [
    ('OLS with p > n = disaster',
     f'With {X.shape[1]} features and {X.shape[0]} countries, OLS achieves perfect R²=1.0 '
     'on training data but catastrophic R² on test data (often below -8). '
     'When there are more features than samples, the system is underdetermined and OLS memorizes rather than learns.',
     RED, 'Module 2'),
    ('Regularization helps but has limits',
     'Ridge and Lasso brought test R² from -8 to ~0.5 by penalizing large coefficients. '
     'But Lasso selected 35 spurious features including numerology scores and flag colors. '
     'Regularization prevents overfitting but does not guarantee meaningful feature selection.',
     '#f59e0b', 'Module 3'),
    ('Random Forest finds the signal',
     'RF concentrated 52% of its feature importance on just 22 causal features (8% of all features), '
     'while giving only 1.5% importance to 92 spurious features. '
     'By randomly subsampling features at each split, RF naturally discovers the most informative variables.',
     GREEN, 'Module 4'),
    ('Always cross-validate',
     'A single train/test split can be misleading — OLS test R² ranged from -20 to +0.5 '
     'depending on the random seed! Cross-validation averages over multiple splits '
     'for reliable performance estimates. RF was consistently the best across all splits.',
     ACCENT_BLUE, 'Module 5'),
    ('No algorithm replaces domain knowledge',
     'Even RF cannot prove causation. It performed well because causal features carry genuine signal. '
     'But in a new domain, you need expertise to separate correlation from causation. '
     'Models find patterns; humans must decide which patterns are meaningful.',
     '#a78bfa', 'All Modules'),
]

cards_html = ''
for title, text, color, module in lessons:
    cards_html += f'''
    <div style="background: #0f172a; padding: 18px 22px; border-radius: 10px;
                border-left: 4px solid {color}; margin: 10px 0; max-width: 900px;">
        <div style="display: flex; justify-content: space-between; align-items: center;">
            <div style="color: {color}; font-size: 16px; font-weight: bold;">{title}</div>
            <div style="color: #64748b; font-size: 12px;">{module}</div>
        </div>
        <div style="color: #cbd5e1; font-size: 14px; margin-top: 8px; line-height: 1.6;">{text}</div>
    </div>'''

display(HTML(cards_html))

In [ ]:
#@title Final Takeaway { display-mode: "form" }

display(takeaway_box(
    '<strong>Random Forest won this competition:</strong> best test R², most stable across splits, '
    'and best at concentrating on causal features. But even RF can\'t prove causation — '
    'it just happens to ignore spurious correlations better than linear models. '
    'Cross-validation confirms these results are reliable, not just luck of the split.'
))